## Importar librerias

In [8]:
from openai import OpenAI
from openai import OpenAIError
from dotenv import load_dotenv
import os

## Configuración del Cliente de OpenAI con Variables de Entorno

In [14]:
# Cargar variables del archivo .env
load_dotenv()

# Obtener la API key desde las variables de entorno
openai_api_key = os.getenv("OPENAI_API_KEY")

# Inicializar cliente (llm) de OpenAI con captura de errores
try:

    llm = OpenAI(api_key=openai_api_key)

    print("El llm de OpenIA se inicializo correctamente. ✅")

    conexion = llm.models.list()

    print("la conexion a la API es correcta. ✅")

except OpenAIError as e:
    print(f"Error al inicializar el llm o conectar a la API: {e}. ❌")

except Exception as e:
    print(f"Error inesperado: {e}. ❌")

El llm de OpenIA se inicializo correctamente. ✅
la conexion a la API es correcta. ✅


## Clasificación de Texto con OpenAI y Modelo GPT-4

In [15]:
# Definir el mensaje de ejemplo
messages = [
    {
        "role": "system",
        "content": "Sos un asistente de clasificación de texto. Responderás a las consultas clasificando el texto proporcionado en las categorías apropiadas."
    },
    {
        "role": "assistant",
        "content": "Las categorías son: Tecnología, Salud, Educación, Finanzas."
    },
    {
        "role": "user",
        "content": "Hoy el Nasdaq sube 3.47%"
    }
]

# Realizar la solicitud a la API
response = llm.chat.completions.create(
    model="gpt-4",
    messages=messages,
    temperature=0.7,
    max_tokens=100,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
)

# Mostrar la respuesta
response.choices[0].message.content

'La categoría apropiada para este texto es: Finanzas.'

## Uso de Mensajes Anteriores a modo de Conversacion OpenAI

In [16]:
mensaje = [
    {"role" : "system",
     "content" : "Sos un asistente de Viajes, planificas y recomendas actividades segun el destino del usuario"
     }
]

print("Hola soy tu asistente de viajes. ¿A donde quieres viajar?")

while True:
    # Entrada del usuario
    usuario_input = input("Yo: ")
    if usuario_input.lower() in ["salir", "chau"]:
        print("Asistente: ¡Buen viaje que lo disfrutes!")
        break

    # Se agrega el input del usuario al historial del chat
    mensaje.append({"role":"user", "content": usuario_input})

    # Imprime la respuesta del usuario

    print(f"Yo: {usuario_input}")

    # se envia el mensaje completo al llm

    try:
        respuesta = llm.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=mensaje,
            temperature=0.75,
            max_tokens=250,
            frequency_penalty=0,
            presence_penalty=0,
        )

        # extrae la respuesta
        respuesta_asistente = respuesta.choices[0].message.content
        print(f"Asistente: {respuesta_asistente}")

        #Agrega la respuesta al historial del chat

        mensaje.append({"role":"assistant", "content":respuesta_asistente})

    except Exception as e:
        print(f"Se produjo un error: {e}")
        break


Hola soy tu asistente de viajes. ¿A donde quieres viajar?
Yo: quiero viajar a San martin de los Andes
Asistente: ¡Qué emoción! San Martín de los Andes es un destino maravilloso en la Patagonia argentina. Aquí te dejo algunas actividades que puedes hacer durante tu visita:

1. **Recorrer la Ruta de los 7 Lagos**: Una de las rutas más pintorescas de la región, que conecta San Martín de los Andes con Villa La Angostura. Puedes hacer paradas en diferentes lagos para disfrutar de hermosos paisajes.

2. **Visitar el Parque Nacional Lanín**: Un lugar ideal para hacer trekking, avistamiento de aves, y disfrutar de la naturaleza en su máxima expresión. También puedes visitar el volcán Lanín.

3. **Pasear por la ciudad**: San Martín de los Andes es un pueblo encantador con calles adoquinadas, casas de madera y una vista increíble al lago Lácar. Puedes recorrer el centro, visitar la plaza principal y disfrutar de la gastronomía local.

4. **Hacer deportes de aventura**: Si te gusta la adrenalina,

## Generación de Audio con OpenAI y Modelo TTS-1

In [21]:
# define el nombre del archivo de audio
archivo_audio_generado = "texto_audio.mp3"

with llm.audio.speech.with_streaming_response.create(
  model="tts-1",
  voice="coral",
  response_format="mp3",
  input=(
        "Este es un audio generado por el modelo TTS-1 y la voz elegida es Coral. "
        "Este modelo es capaz de convertir texto a voz con una calidad increíble. "
        "Puedes usarlo para una variedad de aplicaciones, desde asistentes virtuales "
        "hasta narración de audiolibros y más. La tecnología de síntesis de voz ha avanzado "
        "mucho en los últimos años, y ahora es posible generar voces que suenan muy naturales "
        "y expresivas. Esperamos que disfrutes de esta demostración y veas el potencial que tiene "
        "para transformar la manera en que interactuamos con las máquinas."
    )) as response:
    response.stream_to_file(archivo_audio_generado)

    print(f"Audio guardado en '{archivo_audio_generado}'")

Audio guardado en 'texto_audio.mp3'


## Transcripción de Audio con OpenAI y Modelo Whisper-1

In [22]:
audio_file = open("audio_a_transcribir.mp3", "rb")
transcript = llm.audio.transcriptions.create(
  model="whisper-1",
  file=audio_file
)

texto_en_rosarino = transcript.text
print("Transcripción en idioma original del audio: \n")
print(texto_en_rosarino)

#Traduce transcripción al Frances
# Genera el prompt para el content de GPT

prompt_frances = f"Traduce el siguiente texto en español al frances: '{texto_en_rosarino}'"
traduccion_frances = llm.chat.completions.create(
    model="gpt-4.1",
    messages=[{"role": "user", "content": prompt_frances}]
)

texto_frances = traduccion_frances.choices[0].message.content

print("Trnscripción al Frances: \n")
print(texto_frances)

Transcripción en idioma original del audio: 

¿Cuál es hoy el mejor equipo del mundo para vos? ¿Hay uno? El mejor equipo, si tenemos que decir, es el Madrid, porque es el último campeón de Champions. En los últimos años viene la última, creo que la ganó el City, pero el anterior fueron ellos también. Y un Tourette. Si hablamos por resultados, es el Madrid.
Trnscripción al Frances: 

Bien sûr, aquí tienes la traducción al francés:

« Quel est aujourd’hui la meilleure équipe du monde pour toi ? Y en a-t-il une ? La meilleure équipe, si on doit le dire, c’est Madrid, parce que c’est le dernier champion de la Ligue des Champions. Ces dernières années, la dernière (édition), je crois que c’est City qui l’a gagnée, mais avant c’était eux aussi. Et un Tourette. Si on parle de résultats, c’est Madrid. »

Déjame saber si deseas ajustar alguna parte o necesitas una versión más formal o informal.


## Traducción de Audio con OpenAI y Modelo Whisper-1

In [ ]:
audio_file = open("audio_a_transcribir.mp3", "rb")
transcript = llm.audio.translations.create(
  model="whisper-1",
  file=audio_file
)
transcript

Translation(text='What is the best team in the world for you? Is there one? The best team, if we have to say, is Madrid, because it is the last champion of the Champions League. In recent years, I think the last one was won by City, but the previous one was also them. If we talk about results, it is Madrid.')

## Generación de Respuestas de Chat con GPT-4-Turbo y Análisis de Imágenes

In [ ]:
response = llm.chat.completions.create(
    model="gpt-4-turbo",
    messages=[
        {
          "role": "user",
          "content": [
            {
              "type": "text",
              "text": "Que se puede observar en la imagen?"
            }
          ]
        },
        {
          "role": "user",
          "content": [
            {
              "type": "image_url",
              "image_url": {
                "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
              }
            }
          ]
        }
      ],
    max_tokens=300,
)


# Mostrar la respuesta
response.choices[0].message.content

'En la imagen se puede observar un camino de madera que atraviesa un vasto campo de hierba verde. Este camino proporciona un recorrido que parece extenderse hacia el horizonte, invitando al espectador a explorarlo. El campo está flanqueado por arbustos y algunos árboles dispersos. El cielo sobre el paisaje es azul claro y está decorado con nubes blancas y delgadas, lo que sugiere un día despejado y soleado. La escena transmite una sensación de tranquilidad y conexión con la naturaleza.'